![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [4]:
# =====================================================================
# E19 · Research 输出回传路径探测 · 放入 QC 网页端 Research 的一个 cell 执行
# 任务书外，执行人追加。仅通道探测，不含交易或策略逻辑，不计算收益指标。
# =====================================================================
import os, sys, json, hashlib, platform

MARKER = "probe-marker-E19"
PAYLOAD = (
    "probe-marker-E19\n"
    "line2-ascii\n"
    "line3-中文与全角：测试　保真\n"
    'line4-special: {"json":true} [1,2,3] <tag/> & % $ # @ ! ~ ` ^\n'
    "line5-long: " + "0123456789" * 8 + "\n"
)
PAYLOAD_B = PAYLOAD.encode("utf-8")
PAYLOAD_SHA = hashlib.sha256(PAYLOAD_B).hexdigest()

print("=" * 68)
print("A. 环境指纹")
print("=" * 68)
print("cwd            :", os.getcwd())
print("python         :", sys.version.split()[0])
print("platform       :", platform.platform())
print("libc           :", platform.libc_ver())
print("payload bytes  :", len(PAYLOAD_B))
print("payload sha256 :", PAYLOAD_SHA)

print()
print("=" * 68)
print("B. cwd 目录内容（判断 research 内核是否就在云端项目目录里）")
print("=" * 68)
try:
    for name in sorted(os.listdir(".")):
        p = os.path.join(".", name)
        kind = "DIR " if os.path.isdir(p) else "FILE"
        size = os.path.getsize(p) if os.path.isfile(p) else "-"
        print(f"  {kind} {name}  ({size})")
except Exception as e:
    print("  [FAIL]", type(e).__name__, e)

print()
print("=" * 68)
print("C. 多路径写入测试（哪条能落进云端项目）")
print("=" * 68)
candidates = [
    ("cwd 同级",            f"probe_E19_cwd.txt"),
    ("cwd/子目录",          f"probe_E19_sub/probe_E19_nested.txt"),
    ("上一级",              f"../probe_E19_parent.txt"),
    ("/tmp",                f"/tmp/probe_E19_tmp.txt"),
    ("显式 .ipynb 同名前缀", f"probe_E19_output.json"),
]
write_results = {}
for label, path in candidates:
    try:
        d = os.path.dirname(path)
        if d:
            os.makedirs(d, exist_ok=True)
        data = PAYLOAD_B if not path.endswith(".json") else json.dumps(
            {"marker": MARKER, "sha256": PAYLOAD_SHA, "bytes": len(PAYLOAD_B)},
            ensure_ascii=False, indent=2).encode("utf-8")
        with open(path, "wb") as f:
            f.write(data)
        back = open(path, "rb").read()
        ok = hashlib.sha256(back).hexdigest()
        write_results[label] = {"path": path, "wrote": len(data), "readback_sha256": ok}
        print(f"  [OK]   {label:22} -> {path}  ({len(data)} 字节, 回读 sha256 {ok[:16]}…)")
    except Exception as e:
        write_results[label] = {"path": path, "error": f"{type(e).__name__}: {e}"}
        print(f"  [FAIL] {label:22} -> {path}  {type(e).__name__}: {e}")

print()
print("=" * 68)
print("D. ObjectStore 写入（云端侧，验证方向性）")
print("=" * 68)
try:
    qb = QuantBook()  # noqa: F821  QC 研究环境内置
    qb.object_store.save_bytes("probe_E19_from_research", PAYLOAD_B)
    print("  [OK]   object_store.save_bytes 成功，key = probe_E19_from_research")
    got = qb.object_store.read_bytes("probe_E19_from_research")
    print("  [OK]   同环境内回读成功，", len(got), "字节，sha256",
          hashlib.sha256(bytes(got)).hexdigest()[:16], "…")
except Exception as e:
    print("  [FAIL]", type(e).__name__, e)

print()
print("=" * 68)
print("E. 浏览器下载链接（对照 E17 执行人报告的方式）")
print("=" * 68)
try:
    import base64
    from IPython.display import display, HTML, FileLink
    b64 = base64.b64encode(PAYLOAD_B).decode()
    display(HTML(
        f'<a download="probe_E19_download.txt" '
        f'href="data:text/plain;base64,{b64}">'
        f'▼ 点此下载 probe_E19_download.txt （{len(PAYLOAD_B)} 字节, sha256 {PAYLOAD_SHA[:16]}…）</a>'
    ))
    print("  [OK]   base64 data URI 锚点已生成（上方链接）")
    try:
        display(FileLink("probe_E19_cwd.txt"))
        print("  [OK]   IPython FileLink 已生成")
    except Exception as e:
        print("  [FAIL] FileLink:", type(e).__name__, e)
except Exception as e:
    print("  [FAIL]", type(e).__name__, e)

print()
print("=" * 68)
print("F. 汇总（请连同上方全部输出一起截图回传）")
print("=" * 68)
print(json.dumps({"payload_sha256": PAYLOAD_SHA,
                  "payload_bytes": len(PAYLOAD_B),
                  "writes": write_results}, ensure_ascii=False, indent=2))


In [5]:
import json
cfg = json.load(open("config.json"))
def keys(d, p=""):
    for k, v in (d.items() if isinstance(d, dict) else []):
        print(f"{p}{k}  <{type(v).__name__}>")
        if isinstance(v, dict): keys(v, p + k + ".")
keys(cfg)

In [6]:
# =====================================================================
# E20 · Research → QC file API → 云端项目文件 → lean cloud pull → 本地
# 放入 QC 网页端 Research 的一个新 cell 执行。
# 任务书外，执行人追加。仅通道探测，不含交易或策略逻辑，不计算收益指标。
#
# 凭据纪律：
#   本 cell 使用 airlock/config.json 中 QC 自行注入的 job 凭据，
#   凭据仅在云端内核内存中流转，全程不打印、不写入文件、不外传。
#   所有输出在打印前经 scrub() 过滤，若响应意外回显凭据则替换为 <REDACTED>。
# =====================================================================
import json, hashlib, time

PROJECT_ID = 35174460
FILE_NAME = "probe_E20_output.txt"

PAYLOAD = (
    "probe-marker-E20\n"
    "line2-ascii\n"
    "line3-中文与全角：测试　保真\n"
    'line4-special: {"json":true} [1,2,3] <tag/> & % $ # @ ! ~ ` ^\n'
    "line5-long: " + "0123456789" * 8 + "\n"
)
PAYLOAD_B = PAYLOAD.encode("utf-8")
PAYLOAD_SHA = hashlib.sha256(PAYLOAD_B).hexdigest()

print("payload bytes :", len(PAYLOAD_B))
print("payload sha256:", PAYLOAD_SHA)
print()

cfg = json.load(open("config.json"))
UID = str(cfg.get("job-user-id", ""))
TOK = str(cfg.get("api-access-token", ""))
BASE = str(cfg.get("cloud-api-url", "") or "https://www.quantconnect.com/api/v2/")
if not BASE.endswith("/"):
    BASE += "/"

_SECRETS = [s for s in (TOK, UID) if s]


def scrub(text):
    """在打印前抹掉任何可能回显的凭据。"""
    t = str(text)
    for s in _SECRETS:
        if s:
            t = t.replace(s, "<REDACTED>")
    return t


print("cloud-api-url :", scrub(BASE))
print("job-user-id   : <REDACTED>（长度 %d）" % len(UID))
print("api-access-token 存在:", bool(TOK), "（长度 %d）" % len(TOK))
print()


def qc_post(endpoint, body):
    """按 lean api_client.py:135-152 复现的认证方式发请求。"""
    import requests
    ts = str(int(time.time()))
    pw = hashlib.sha256(f"{TOK}:{ts}".encode("utf-8")).hexdigest()
    r = requests.post(
        BASE + endpoint,
        headers={"Timestamp": ts},
        auth=(UID, pw),
        json=body,
        timeout=60,
    )
    return r.status_code, r.text


print("=" * 68)
print("1. files/create —— 尝试把结果写成云端项目文件")
print("=" * 68)
try:
    code, text = qc_post("files/create", {
        "projectId": PROJECT_ID,
        "name": FILE_NAME,
        "content": PAYLOAD,
    })
    print("HTTP", code)
    print(scrub(text)[:800])
except Exception as e:
    print("[FAIL]", type(e).__name__, scrub(e))

print()
print("=" * 68)
print("2. files/update —— 若 create 因已存在而失败，走更新")
print("=" * 68)
try:
    code, text = qc_post("files/update", {
        "projectId": PROJECT_ID,
        "name": FILE_NAME,
        "content": PAYLOAD,
    })
    print("HTTP", code)
    print(scrub(text)[:800])
except Exception as e:
    print("[FAIL]", type(e).__name__, scrub(e))

print()
print("=" * 68)
print("3. files/read —— 回读核证，确认云端已收到")
print("=" * 68)
try:
    code, text = qc_post("files/read", {"projectId": PROJECT_ID, "name": FILE_NAME})
    print("HTTP", code)
    body = scrub(text)
    print(body[:800])
    try:
        j = json.loads(text)
        for f in (j.get("files") or []):
            c = f.get("content", "")
            print("回读 content 字节:", len(c.encode("utf-8")))
            print("回读 content sha256:", hashlib.sha256(c.encode("utf-8")).hexdigest())
            print("与载荷一致:", hashlib.sha256(c.encode("utf-8")).hexdigest() == PAYLOAD_SHA)
    except Exception:
        pass
except Exception as e:
    print("[FAIL]", type(e).__name__, scrub(e))

print()
print("=" * 68)
print("4. storage-permissions（config.json 中的 ObjectStore 权限声明）")
print("=" * 68)
print(json.dumps(cfg.get("storage-permissions"), ensure_ascii=False))


In [7]:
# =====================================================================
# E20 · Research → QC file API → 云端项目文件 → lean cloud pull → 本地
# 放入 QC 网页端 Research 的一个新 cell 执行。
# 任务书外，执行人追加。仅通道探测，不含交易或策略逻辑，不计算收益指标。
#
# 凭据纪律：
#   本 cell 使用 airlock/config.json 中 QC 自行注入的 job 凭据，
#   凭据仅在云端内核内存中流转，全程不打印、不写入文件、不外传。
#   所有输出在打印前经 scrub() 过滤，若响应意外回显凭据则替换为 <REDACTED>。
# =====================================================================
import json, hashlib, time

PROJECT_ID = 35174460
FILE_NAME = "probe_E20_output.txt"

PAYLOAD = (
    "probe-marker-E20\n"
    "line2-ascii\n"
    "line3-中文与全角：测试　保真\n"
    'line4-special: {"json":true} [1,2,3] <tag/> & % $ # @ ! ~ ` ^\n'
    "line5-long: " + "0123456789" * 8 + "\n"
)
PAYLOAD_B = PAYLOAD.encode("utf-8")
PAYLOAD_SHA = hashlib.sha256(PAYLOAD_B).hexdigest()

print("payload bytes :", len(PAYLOAD_B))
print("payload sha256:", PAYLOAD_SHA)
print()

cfg = json.load(open("config.json"))
UID = str(cfg.get("job-user-id", ""))
TOK = str(cfg.get("api-access-token", ""))

# 首轮错误更正：config.json 的 cloud-api-url 实测为
#   https://www.quantconnect.com/api/v2/cloud/     ← 带 /cloud/ 后缀
# 直接拼 files/create 会得到 api/v2/cloud/files/create，该端点不存在。
# 项目与文件 API 的正确 base 是 api/v2/（不含 cloud/），
# 与 lean/constants.py:85 的 API_BASE_URL 一致。
_RAW = str(cfg.get("cloud-api-url", "") or "")
if not _RAW.endswith("/"):
    _RAW += "/"
BASE = _RAW[: -len("cloud/")] if _RAW.endswith("cloud/") else _RAW
if not BASE:
    BASE = "https://www.quantconnect.com/api/v2/"

# 两个 base 都试，一轮内定论
BASES = [("api/v2（正确候选）", BASE)]
if _RAW and _RAW != BASE:
    BASES.append(("api/v2/cloud（首轮误用，作对照）", _RAW))

_SECRETS = [s for s in (TOK, UID) if s]


def scrub(text):
    """在打印前抹掉任何可能回显的凭据。"""
    t = str(text)
    for s in _SECRETS:
        if s:
            t = t.replace(s, "<REDACTED>")
    return t


print("job-user-id   : <REDACTED>（长度 %d）" % len(UID))
print("api-access-token 存在:", bool(TOK), "（长度 %d）" % len(TOK))
for label, b in BASES:
    print(f"base [{label}] : {scrub(b)}")
print()


def qc_post(base, endpoint, body):
    """按 lean api_client.py:135-152 复现的认证方式发请求。"""
    import requests
    ts = str(int(time.time()))
    pw = hashlib.sha256(f"{TOK}:{ts}".encode("utf-8")).hexdigest()
    r = requests.post(
        base + endpoint,
        headers={"Timestamp": ts},
        auth=(UID, pw),
        json=body,
        timeout=60,
    )
    return r.status_code, r.text


for label, base in BASES:
    print("#" * 68)
    print(f"# base = {scrub(base)}   [{label}]")
    print("#" * 68)

    for step, endpoint, body in [
        ("1. files/create", "files/create",
         {"projectId": PROJECT_ID, "name": FILE_NAME, "content": PAYLOAD}),
        ("2. files/update", "files/update",
         {"projectId": PROJECT_ID, "name": FILE_NAME, "content": PAYLOAD}),
        ("3. files/read", "files/read",
         {"projectId": PROJECT_ID, "name": FILE_NAME}),
    ]:
        print("=" * 68)
        print(step)
        print("=" * 68)
        try:
            code, text = qc_post(base, endpoint, body)
            print("HTTP", code)
            print(scrub(text)[:800])
            if endpoint == "files/read":
                try:
                    j = json.loads(text)
                    for f in (j.get("files") or []):
                        c = f.get("content", "")
                        h = hashlib.sha256(c.encode("utf-8")).hexdigest()
                        print("回读 content 字节 :", len(c.encode("utf-8")))
                        print("回读 content sha256:", h)
                        print(">>> 与载荷一致:", h == PAYLOAD_SHA, "<<<")
                except Exception:
                    pass
        except Exception as e:
            print("[FAIL]", type(e).__name__, scrub(e))
        print()

print()
print("=" * 68)
print("4. storage-permissions（config.json 中的 ObjectStore 权限声明）")
print("=" * 68)
print(json.dumps(cfg.get("storage-permissions"), ensure_ascii=False))
